# Stage F Forensic Comparison

Compare a read-only historical Notebook 51R2 run with a separate fresh Notebook 51R2 rerun. Edit only `HISTORICAL_N51` and `RERUN_N51` in the code cell.

This notebook does not regenerate Notebook 51R2. The fresh rerun must already exist in a distinct output directory.


In [ ]:
# ============================================================
# STAGE F FORENSIC COMPARISON - COMPLETE CORRECTED CELL
#
# PURPOSE
# -------
# Compare a read-only historical Notebook 51R2 run with a
# separate, fresh Notebook 51R2 rerun. This avoids the previous
# same-directory, checkpoint-reuse and candidate-filtering errors.
#
# PRECONDITION
# ------------
# You must have TWO DISTINCT completed run directories:
#
#   HISTORICAL_N51 = the original completed 51R2 golden-master run
#   RERUN_N51      = a newly generated 51R2 run created in a fresh
#                    output directory under the current runtime
#
# Do not point both variables to the same directory.
# Do not rerun candidate construction inside the historical run.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys

import cv2
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. EDIT ONLY THESE TWO PATHS
# ------------------------------------------------------------

HISTORICAL_N51 = Path(
    '/content/drive/MyDrive/MusicAnnotationProject/'
    'observed_print_classifier_v8_7_notebook51/'
    'REPLACE_WITH_HISTORICAL_51R2_RUN'
)

RERUN_N51 = Path(
    '/content/drive/MyDrive/MusicAnnotationProject/'
    'observed_print_classifier_v8_7_notebook51/'
    'REPLACE_WITH_FRESH_51R2_RERUN'
)

# Optional. Leave as None to create the audit beside the fresh rerun.
AUDIT_ROOT_OVERRIDE = None

ARM = 'M5_REMOVE'
RESTORER = 'R2_telea'
EXPECTED_FILENAME = 'M5_R2_telea.png'


# ------------------------------------------------------------
# 2. Helpers
# ------------------------------------------------------------

def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256_file(path):
    path = Path(path)
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1048576), b''):
            h.update(block)
    return h.hexdigest()


def array_hash(array):
    array = np.ascontiguousarray(array)
    h = hashlib.sha256()
    h.update(str(array.dtype).encode())
    h.update(str(array.shape).encode())
    h.update(array.tobytes())
    return h.hexdigest()


def load_gray(path, description):
    path = Path(path)
    require(path.exists(), f'Missing {description}: {path}')
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    require(image is not None, f'Unreadable {description}: {path}')
    return image


def unique_inventory(run_root):
    run_root = Path(run_root)

    preferred = run_root / '04_candidate_lock' / 'candidate_inventory.csv'
    if preferred.exists():
        return preferred

    hits = sorted(
        p for p in run_root.rglob('candidate_inventory.csv')
        if p.is_file()
    )

    require(
        len(hits) == 1,
        f'Expected one candidate_inventory.csv beneath {run_root}; '
        f'found {len(hits)}: {hits}'
    )

    return hits[0]


def load_inventory(run_root):
    path = unique_inventory(run_root)
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)

    required = {
        'page',
        'arm',
        'restorer',
        'candidate',
    }

    missing = required - set(frame.columns)
    require(
        not missing,
        f'Candidate inventory {path} lacks columns: {sorted(missing)}'
    )

    for column in required:
        frame[column] = frame[column].astype(str).str.strip()

    selected = frame[
        (frame.arm == ARM)
        &
        (frame.restorer == RESTORER)
    ].copy()

    require(
        not selected.empty,
        f'No {ARM}/{RESTORER} rows in {path}. '
        f'Arm values: {sorted(frame.arm.unique().tolist())}; '
        f'restorer values: {sorted(frame.restorer.unique().tolist())}'
    )

    require(
        not selected.page.duplicated().any(),
        f'Duplicate {ARM}/{RESTORER} page rows in {path}: '
        f'{sorted(selected.loc[selected.page.duplicated(False), "page"].unique())}'
    )

    return path, selected


def resolve_candidate(run_root, row):
    run_root = Path(run_root)
    recorded = Path(str(row['candidate']))

    # Use the recorded path only if it is inside the requested run.
    if recorded.exists():
        try:
            recorded.resolve().relative_to(run_root.resolve())
            return recorded
        except ValueError:
            pass

    page = str(row['page'])
    filename = recorded.name or EXPECTED_FILENAME

    direct = run_root / '03_candidates' / page / filename
    if direct.exists():
        return direct

    expected = run_root / '03_candidates' / page / EXPECTED_FILENAME
    if expected.exists():
        return expected

    hits = sorted(
        p for p in run_root.rglob(filename)
        if p.is_file() and p.parent.name == page
    )

    require(
        len(hits) == 1,
        f'{page}: expected one candidate named {filename} beneath '
        f'{run_root}; found {len(hits)}: {hits}'
    )

    return hits[0]


# ------------------------------------------------------------
# 3. Validate that the runs are genuinely distinct
# ------------------------------------------------------------

HISTORICAL_N51 = HISTORICAL_N51.expanduser().resolve()
RERUN_N51 = RERUN_N51.expanduser().resolve()

require(
    HISTORICAL_N51.exists(),
    f'Historical run does not exist: {HISTORICAL_N51}'
)

require(
    RERUN_N51.exists(),
    f'Fresh rerun does not exist: {RERUN_N51}'
)

require(
    HISTORICAL_N51 != RERUN_N51,
    'Historical and rerun directories are identical. '
    'Create the rerun in a fresh Notebook51R2 output directory.'
)

try:
    RERUN_N51.relative_to(HISTORICAL_N51)
    raise RuntimeError(
        'Rerun directory is nested inside the historical run. '
        'Use two independent run directories.'
    )
except ValueError:
    pass

try:
    HISTORICAL_N51.relative_to(RERUN_N51)
    raise RuntimeError(
        'Historical directory is nested inside the rerun. '
        'Use two independent run directories.'
    )
except ValueError:
    pass


# ------------------------------------------------------------
# 4. Create a separate audit directory
# ------------------------------------------------------------

if AUDIT_ROOT_OVERRIDE is None:
    audit_parent = RERUN_N51.parent / 'stage_f_forensic_audits'
else:
    audit_parent = Path(AUDIT_ROOT_OVERRIDE)

run_id = datetime.now().strftime('%Y%m%d_%H%M%S') + '_stage_f_51r2_rerun_comparison'
AUDIT_ROOT = audit_parent / run_id
TABLES = AUDIT_ROOT / 'tables'
DIFFS = AUDIT_ROOT / 'difference_images'

for directory in [AUDIT_ROOT, TABLES, DIFFS]:
    directory.mkdir(parents=True, exist_ok=False)


# ------------------------------------------------------------
# 5. Load and validate both candidate inventories
# ------------------------------------------------------------

historical_inventory_path, historical = load_inventory(HISTORICAL_N51)
rerun_inventory_path, rerun = load_inventory(RERUN_N51)

historical_pages = set(historical.page)
rerun_pages = set(rerun.page)

require(
    historical_pages == rerun_pages,
    'Historical/rerun page rosters differ. '
    f'Missing from rerun: {sorted(historical_pages - rerun_pages)}; '
    f'extra in rerun: {sorted(rerun_pages - historical_pages)}'
)

print('=' * 80)
print('STAGE F NOTEBOOK51R2 RERUN COMPARISON')
print('=' * 80)
print('Historical run:', HISTORICAL_N51)
print('Fresh rerun:', RERUN_N51)
print('Historical inventory:', historical_inventory_path)
print('Rerun inventory:', rerun_inventory_path)
print('Pages:', len(historical_pages))
print('Audit output:', AUDIT_ROOT)


# ------------------------------------------------------------
# 6. Compare the two runs page by page
# ------------------------------------------------------------

comparison_rows = []
coordinate_rows = []

historical_lookup = historical.set_index('page', drop=False)
rerun_lookup = rerun.set_index('page', drop=False)

for page in sorted(historical_pages):
    historical_row = historical_lookup.loc[page]
    rerun_row = rerun_lookup.loc[page]

    historical_path = resolve_candidate(HISTORICAL_N51, historical_row)
    rerun_path = resolve_candidate(RERUN_N51, rerun_row)

    require(
        historical_path.resolve() != rerun_path.resolve(),
        f'{page}: historical and rerun resolve to the same candidate file: '
        f'{historical_path}'
    )

    historical_image = load_gray(
        historical_path,
        f'{page} historical candidate'
    )

    rerun_image = load_gray(
        rerun_path,
        f'{page} rerun candidate'
    )

    require(
        historical_image.shape == rerun_image.shape,
        f'{page}: shape mismatch: historical {historical_image.shape}, '
        f'rerun {rerun_image.shape}'
    )

    different = historical_image != rerun_image
    different_pixels = int(different.sum())

    signed = (
        rerun_image.astype(np.int16)
        - historical_image.astype(np.int16)
    )

    absolute = np.abs(signed).astype(np.uint8)
    exact = different_pixels == 0

    page_diff_dir = DIFFS / page
    page_diff_dir.mkdir(parents=True, exist_ok=False)

    mask_image = different.astype(np.uint8) * 255
    require(
        cv2.imwrite(str(page_diff_dir / 'difference_mask.png'), mask_image),
        f'{page}: could not write difference mask'
    )
    require(
        cv2.imwrite(str(page_diff_dir / 'absolute_difference.png'), absolute),
        f'{page}: could not write absolute difference'
    )

    historical_file_sha = sha256_file(historical_path)
    rerun_file_sha = sha256_file(rerun_path)
    historical_array_sha = array_hash(historical_image)
    rerun_array_sha = array_hash(rerun_image)

    historical_locked_file_sha = str(historical_row.get('sha256', '')).strip()
    historical_locked_array_sha = str(historical_row.get('array_sha256', '')).strip()
    rerun_locked_file_sha = str(rerun_row.get('sha256', '')).strip()
    rerun_locked_array_sha = str(rerun_row.get('array_sha256', '')).strip()

    comparison_rows.append({
        'page': page,
        'exact': exact,
        'different_pixels': different_pixels,
        'maximum_absolute_difference': int(absolute.max()),
        'mean_absolute_difference_on_mismatch': (
            float(absolute[different].mean()) if different_pixels else 0.0
        ),
        'historical_path': str(historical_path),
        'rerun_path': str(rerun_path),
        'historical_file_sha256': historical_file_sha,
        'rerun_file_sha256': rerun_file_sha,
        'historical_array_sha256': historical_array_sha,
        'rerun_array_sha256': rerun_array_sha,
        'historical_inventory_file_sha256': historical_locked_file_sha or None,
        'historical_inventory_array_sha256': historical_locked_array_sha or None,
        'rerun_inventory_file_sha256': rerun_locked_file_sha or None,
        'rerun_inventory_array_sha256': rerun_locked_array_sha or None,
        'historical_file_lock_valid': (
            historical_file_sha == historical_locked_file_sha
            if historical_locked_file_sha else None
        ),
        'historical_array_lock_valid': (
            historical_array_sha == historical_locked_array_sha
            if historical_locked_array_sha else None
        ),
        'rerun_file_lock_valid': (
            rerun_file_sha == rerun_locked_file_sha
            if rerun_locked_file_sha else None
        ),
        'rerun_array_lock_valid': (
            rerun_array_sha == rerun_locked_array_sha
            if rerun_locked_array_sha else None
        ),
    })

    ys, xs = np.where(different)
    for y, x in zip(ys, xs):
        coordinate_rows.append({
            'page': page,
            'y': int(y),
            'x': int(x),
            'historical_value': int(historical_image[y, x]),
            'rerun_value': int(rerun_image[y, x]),
            'signed_difference': int(signed[y, x]),
            'absolute_difference': int(absolute[y, x]),
        })

    print(
        f'{page}: exact={exact}, diff={different_pixels}, '
        f'max_abs={int(absolute.max())}'
    )


# ------------------------------------------------------------
# 7. Write evidence tables
# ------------------------------------------------------------

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    ['different_pixels', 'page'],
    ascending=[False, True],
).reset_index(drop=True)

coordinate_columns = [
    'page',
    'y',
    'x',
    'historical_value',
    'rerun_value',
    'signed_difference',
    'absolute_difference',
]
coordinate_df = pd.DataFrame(coordinate_rows, columns=coordinate_columns)

comparison_path = TABLES / 'historical_vs_fresh_51r2_candidates.csv'
coordinate_path = TABLES / 'historical_vs_fresh_51r2_difference_coordinates.csv'

comparison_df.to_csv(comparison_path, index=False)
coordinate_df.to_csv(coordinate_path, index=False)


# ------------------------------------------------------------
# 8. Formal verdict
# ------------------------------------------------------------

all_exact = bool(comparison_df.exact.all())
total_difference = int(comparison_df.different_pixels.sum())
failed_pages = comparison_df.loc[~comparison_df.exact, 'page'].tolist()

historical_locks_valid = bool(
    comparison_df['historical_file_lock_valid'].dropna().all()
    and comparison_df['historical_array_lock_valid'].dropna().all()
)
rerun_locks_valid = bool(
    comparison_df['rerun_file_lock_valid'].dropna().all()
    and comparison_df['rerun_array_lock_valid'].dropna().all()
)

if all_exact:
    diagnosis = 'CURRENT_RUNTIME_REPRODUCES_HISTORICAL_51R2_GOLDEN_MASTER'
    required_action = (
        'The current runtime can reproduce Notebook51R2 exactly. '
        'Stage F should be fixed by making Notebook53R2 use the exact '
        'Notebook51R2 observed input, regional mask construction, region '
        'ordering and composition logic. Do not introduce a tolerance.'
    )
else:
    diagnosis = 'CURRENT_RUNTIME_DOES_NOT_REPRODUCE_HISTORICAL_51R2_GOLDEN_MASTER'
    required_action = (
        'Stop changing Notebook53R2 reconstruction logic. The historical '
        'Notebook51R2 golden master is not reproduced by a fresh run under '
        'the current runtime/state. Recover the historical software/input '
        'environment or define a separately governed reference-locked replay.'
    )

verdict = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'experiment': 'Historical Notebook51R2 versus fresh Notebook51R2 rerun',
    'historical_run': str(HISTORICAL_N51),
    'fresh_rerun': str(RERUN_N51),
    'historical_inventory': str(historical_inventory_path),
    'rerun_inventory': str(rerun_inventory_path),
    'arm': ARM,
    'restorer': RESTORER,
    'pages_compared': int(len(comparison_df)),
    'exact_pages': int(comparison_df.exact.sum()),
    'failed_pages': failed_pages,
    'total_different_pixels': total_difference,
    'maximum_absolute_difference': int(
        comparison_df.maximum_absolute_difference.max()
    ),
    'historical_inventory_locks_valid': historical_locks_valid,
    'rerun_inventory_locks_valid': rerun_locks_valid,
    'fresh_run_reproduces_historical_golden_master': all_exact,
    'diagnosis': diagnosis,
    'required_action': required_action,
    'stage_f_may_be_declared_pass': False,
    'stage_f_pass_condition': (
        'Notebook53R2 corrected pages must equal the locked historical '
        'Notebook51R2 corrected pages with zero differing pixels.'
    ),
    'software': {
        'python': sys.version,
        'opencv': cv2.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'platform': platform.platform(),
    },
}

verdict_path = AUDIT_ROOT / 'stage_f_forensic_verdict.json'
verdict_path.write_text(json.dumps(verdict, indent=2))

(AUDIT_ROOT / 'software_environment.json').write_text(
    json.dumps(verdict['software'], indent=2)
)

summary_lines = [
    'STAGE F FORENSIC COMPARISON',
    '',
    f'Historical run: {HISTORICAL_N51}',
    f'Fresh rerun: {RERUN_N51}',
    f'Pages compared: {len(comparison_df)}',
    f'Exact pages: {int(comparison_df.exact.sum())}',
    f'Failed pages: {failed_pages}',
    f'Total differing pixels: {total_difference}',
    f'Maximum absolute difference: {int(comparison_df.maximum_absolute_difference.max())}',
    '',
    f'Diagnosis: {diagnosis}',
    '',
    required_action,
    '',
    'This audit does not itself make Notebook54 Stage F pass.',
]

(AUDIT_ROOT / 'executive_summary.txt').write_text('\n'.join(summary_lines))

print('\n' + '=' * 80)
print('FORENSIC VERDICT')
print('=' * 80)
print(json.dumps(verdict, indent=2))
print('\nOutputs:')
print(' ', comparison_path)
print(' ', coordinate_path)
print(' ', verdict_path)
print(' ', DIFFS)
